# Dataset Collection Control Panel

Use this notebook to start a dataset collection run and monitor progress from `state.json` and collector log files.

Notes:
- the public osu! rankings endpoint currently exposes only the top `10000` users;
- the collector is resumable when you restart it with the same `run_name` and config;
- each run directory stores `config.json`, `state.json`, `collector.stdout.log`, `collector.stderr.log`, raw JSONL files, the flattened CSV, and `profiling_summary.md`.


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, clear_output, display


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'scripts' / 'collect_osu_ranked_dataset.py').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from the current notebook environment.')


REPO_ROOT = find_repo_root(Path.cwd())
RAW_DIR = REPO_ROOT / 'data' / 'raw'
COLLECTOR = REPO_ROOT / 'scripts' / 'collect_osu_ranked_dataset.py'

display(Markdown(f'**Repo root:** `{REPO_ROOT}`'))
display(Markdown(f'**Collector:** `{COLLECTOR}`'))


In [ ]:
def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


def build_band_spec(sample_per_band: int = 100) -> str:
    bands = [
        '1-1000',
        '1001-2000',
        '2001-3000',
        '3001-4000',
        '4001-5000',
        '5001-6000',
        '6001-7000',
        '7001-8000',
        '8001-9000',
        '9001-10000',
    ]
    return ','.join(f'{band}:{sample_per_band}' for band in bands)


RUN_NAME = f'osu_try_data_full_{utc_stamp()}'
BAND_SPEC = build_band_spec(sample_per_band=100)
RECENT_SCORES_PER_USER = 30
BEST_SCORES_PER_USER = 20
REQUEST_DELAY_SECONDS = 0.05
RANDOM_SEED = 42

RUN_DIR = RAW_DIR / RUN_NAME
RUN_DIR


In [ ]:
def build_command(run_name: str) -> list[str]:
    output_dir = RAW_DIR / run_name
    return [
        'python',
        str(COLLECTOR),
        '--output-dir',
        str(output_dir),
        '--band-spec',
        BAND_SPEC,
        '--recent-scores-per-user',
        str(RECENT_SCORES_PER_USER),
        '--best-scores-per-user',
        str(BEST_SCORES_PER_USER),
        '--random-seed',
        str(RANDOM_SEED),
        '--request-delay-seconds',
        str(REQUEST_DELAY_SECONDS),
    ]


def start_collection(run_name: str) -> Path:
    run_dir = RAW_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = run_dir / 'collector.stdout.log'
    stderr_path = run_dir / 'collector.stderr.log'
    command = build_command(run_name)

    stdout_handle = stdout_path.open('a', encoding='utf-8')
    stderr_handle = stderr_path.open('a', encoding='utf-8')
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=stdout_handle,
        stderr=stderr_handle,
        creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0),
    )
    print('Started collector PID:', process.pid)
    print('Run dir:', run_dir)
    print('Command:')
    print(' '.join(shlex.quote(part) for part in command))
    return run_dir


# Example: uncomment the next line when you want to start a new run.
# RUN_DIR = start_collection(RUN_NAME)


In [ ]:
def read_json(path: Path, default):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def tail_lines(path: Path, limit: int = 10) -> list[str]:
    if not path.exists():
        return []
    lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    return lines[-limit:]


def render_progress(run_dir: Path, log_tail: int = 12) -> str:
    state = read_json(run_dir / 'state.json', {})
    metadata = read_json(run_dir / 'export_metadata.json', {})
    config = read_json(run_dir / 'config.json', {})
    stdout_tail = tail_lines(run_dir / 'collector.stdout.log', limit=log_tail)
    stderr_tail = tail_lines(run_dir / 'collector.stderr.log', limit=log_tail)

    phase = state.get('phase', 'unknown')
    processed = state.get('processed_user_count', 0)
    total = state.get('total_sampled_user_count') or state.get('sampled_user_count') or 0
    progress = (processed / total * 100.0) if total else 0.0

    lines = [
        '# Collection Progress',
        '',
        f'- Run dir: `{run_dir}`',
        f'- Phase: `{phase}`',
        f'- Last updated: `{state.get("last_updated_at", "")}`',
        f'- Ranking type: `{state.get("ranking_type") or config.get("ranking_type", "")}`',
        f'- Ranking total available: `{state.get("ranking_total_available", "")}`',
        f'- Users processed: `{processed}` / `{total}` ({progress:.2f}%)',
        f'- Unique sampled users: `{state.get("unique_sampled_user_count", "")}`',
        f'- Ranking pages fetched: `{state.get("ranking_pages_fetched", "")}` / `{state.get("ranking_pages_total_available", "")}`',
        f'- Beatmaps cached: `{state.get("cached_beatmap_count", "")}`',
        f'- Beatmaps referenced: `{state.get("referenced_beatmap_count", "")}`',
        f'- CSV rows: `{state.get("csv_row_count", metadata.get("exported_row_count", ""))}`',
        f'- Current band: `{state.get("current_band_label", "")}`',
        f'- Current page: `{state.get("current_page", "")}`',
        f'- Note: `{state.get("note", "")}`',
        '',
        '## Sampled Users Per Band',
        '',
    ]

    sampled_band_counts = state.get('sampled_band_counts') or {}
    processed_band_counts = state.get('processed_band_counts') or {}
    if sampled_band_counts:
        lines.extend(f'- `{band}`: `{count}`' for band, count in sorted(sampled_band_counts.items()))
    else:
        lines.append('- no data yet')

    lines.extend(['', '## Processed Users Per Band', ''])
    if processed_band_counts:
        lines.extend(f'- `{band}`: `{count}`' for band, count in sorted(processed_band_counts.items()))
    else:
        lines.append('- no data yet')

    lines.extend(['', '## Collector Stdout Tail', ''])
    if stdout_tail:
        lines.extend(f'- `{line}`' for line in stdout_tail)
    else:
        lines.append('- no stdout log yet')

    lines.extend(['', '## Collector Stderr Tail', ''])
    if stderr_tail:
        lines.extend(f'- `{line}`' for line in stderr_tail)
    else:
        lines.append('- no stderr log yet')

    return '\n'.join(lines)


def show_progress(run_dir: Path) -> None:
    display(Markdown(render_progress(run_dir)))


def watch_progress(run_dir: Path, refresh_seconds: float = 5.0, max_refreshes: int | None = None) -> None:
    iteration = 0
    while True:
        clear_output(wait=True)
        show_progress(run_dir)
        state = read_json(run_dir / 'state.json', {})
        if state.get('phase') == 'done':
            break
        iteration += 1
        if max_refreshes is not None and iteration >= max_refreshes:
            break
        time.sleep(refresh_seconds)


# Example for an existing run:
# RUN_DIR = RAW_DIR / 'my_run_name'
# show_progress(RUN_DIR)
# watch_progress(RUN_DIR, refresh_seconds=5)


## Typical workflow

1. Adjust `RUN_NAME`, `BAND_SPEC`, and score limits in the config cell.
2. Run `RUN_DIR = start_collection(RUN_NAME)`.
3. Run `watch_progress(RUN_DIR, refresh_seconds=5)` in the next cell.
4. When the run finishes, open `profiling_summary.md` and the CSV from the run directory.

If a run stops unexpectedly, keep the same `RUN_NAME` and start it again. The collector uses the same run directory and resumes from `state.json` plus the append-only raw JSONL files.
